In [1]:
import torch
from diffusers import ZImagePipeline

pipe = ZImagePipeline.from_pretrained(
    "Tongyi-MAI/Z-Image",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=False,
)

/home/ubuntu/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1021 06:29:31.268000 14507 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1021 06:29:31.299000 14507 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
/home/ubuntu/miniconda3/lib/python3.13/site-package

In [2]:
pipe = pipe.to("cuda")

In [17]:
import re
import wandb
import judge

def extract(text):
  if "</think>" in text:
    return text.split("</think>")[1]
  else:
    return text


def gen_image(prompt):
  return pipe(
      prompt=prompt.split("%%")[0],
      negative_prompt=prompt.split("%%")[1] if "%%" in prompt else "",
      height=768,
      width=768,
      cfg_normalization=False,
      num_inference_steps=50,
      guidance_scale=3.0,
      generator=torch.Generator("cuda").manual_seed(42),
  ).images[0]

past = []

def format_reward(completions, log_extra=None, log_metric=None, **kwargs):
  rewards = []
  for com in completions:
    r_0 = (com[-1]["content"].count("%%") == 1) # has both positive and negative prompt
    r_1 = ("prompt" not in com[-1]["content"]) * 1 # does not leak the prompt label into the text
    r_2 = (len(com[-1]["content"]) <= 500) * 1 # cannot be too long or runaway
    rewards.append(r_0 + r_1 + r_2)
  return rewards

def reward_func_text(completions, log_extra=None, log_metric=None, **kwargs):
  global past
  rewards = [judge.text_reward(extract(completion[-1]["content"]), past) for completion in completions]
  past.extend([extract(i[-1]["content"].split("%%")[0]) for i in completions])
  print(rewards)
  if len(past) > 200:
    past = past[-200:]
  return [i for i in rewards]


In [18]:
judge.text_reward("an dark, gale like green, over exposed photo of an apple, the apple is rotten %% normal color", ["an dark, gale like orange, over exposed photo of an apple, the apple is rotten %% normal color"]) 

0.22000000000000003

In [19]:
!wget https://weathon.github.io/icml2026_position/site/img/asserts/real/2235.jpg -O test.jpg

--2026-10-21 06:31:07--  https://weathon.github.io/icml2026_position/site/img/asserts/real/2235.jpg
Resolving weathon.github.io (weathon.github.io)... 185.199.109.153, 185.199.108.153, 185.199.111.153, ...
Connecting to weathon.github.io (weathon.github.io)|185.199.109.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 24399 (24K) [image/jpeg]
Saving to: ‘test.jpg’

test.jpg            100%[===================>]  23.83K  --.-KB/s    in 0.03s   

2026-10-21 06:31:07 (704 KB/s) - ‘test.jpg’ saved [24399/24399]



In [20]:
judge.image_reward("test.jpg", "nature")

0.47500000000000003

In [41]:
import torch
from transformers import AutoImageProcessor, AutoModel
from transformers.image_utils import load_image


pretrained_model_name = "facebook/dinov3-convnext-tiny-pretrain-lvd1689m"
dino_processor = AutoImageProcessor.from_pretrained(pretrained_model_name)
dino = AutoModel.from_pretrained(
    pretrained_model_name,
    device_map="cuda",
)

Loading weights: 100%|██████████| 180/180 [00:00<00:00, 4951.21it/s]


In [42]:
def extract_features(image_path):
    image = load_image(image_path).convert("RGB") 
    inputs = dino_processor(images=image, return_tensors="pt").to(dino.device)
    with torch.inference_mode():
        outputs = dino(**inputs)

    pooled_output = outputs.pooler_output
    return pooled_output

In [44]:
torch.nn.functional.cosine_similarity(torch.tensor(extract_features("test.jpg")), torch.tensor(extract_features("test2.png")), dim=1)

/tmp/ipykernel_14507/1221493245.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.nn.functional.cosine_similarity(torch.tensor(extract_features("test.jpg")), torch.tensor(extract_features("test2.png")), dim=1)


tensor([0.7526], device='cuda:0')

In [45]:
import numpy as np

def _normalize(x):
    x = np.atleast_2d(np.asarray(x, dtype=np.float32))
    return x / np.linalg.norm(x, axis=1, keepdims=True)

def _pick_k(n, frac=0.01, k_min=20):
    return min(n, max(k_min, int(round(frac * n))))

def topk_cos_score(bank, v, frac=0.01, k_min=20):
    bank = np.array(bank)
    B, V = _normalize(bank), _normalize(v)
    k = _pick_k(len(B), frac, k_min)
    sims = V @ B.T
    s = np.partition(sims, -k, axis=1)[:, -k:].mean(axis=1)
    return s[0] if np.ndim(v) == 1 else s

def bank_self_scores(bank, frac=0.01, k_min=20):
    B = _normalize(bank)
    k = _pick_k(len(B) - 1, frac, k_min)
    S = B @ B.T
    np.fill_diagonal(S, -np.inf)
    return np.partition(S, -k, axis=1)[:, -k:].mean(axis=1)

In [46]:
v_bank = [np.random.rand(768)]
index = 0
def reward_func_image(completions, product, log_extra=None, log_metric=None, **kwargs):
  global index, v_bank
  prompts = [extract(completion[-1]["content"]) for completion in completions]
  images = [gen_image(prompt) for prompt in prompts]
  image_paths = []
  for i in range(len(images)):
      images[i].save(f"images/{index:05d}.png")
      image_paths.append(f"images/{index:05d}.png")
      index += 1 
  rewards = [judge.image_reward(path, product[0]) for path in image_paths]
  diversity_scores = []
  for path in image_paths:
    v = extract_features(path).cpu().numpy()[0] 
    diversity_scores.append(1 - topk_cos_score(v_bank, v))
    v_bank.append(v)
  print(diversity_scores)
  for i in range(len(prompts)):
    if wandb.run is not None:
      wandb.log({
          "image": wandb.Image(images[i], caption=f"reward: {rewards[i]}, diversity_scores: {diversity_scores[i]}, product: {product[0]}")
      })
  print(rewards)
  return [(i + j) for i, j in zip(rewards, diversity_scores)]

In [47]:
reward_func_image([ [{"content": "a apple tree"}] ], ["toilet"])

  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:08<00:00,  6.11it/s]


[np.float32(0.99589723)]
[0.006666666666666667]


[np.float32(1.002564)]

In [48]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    learning_rate=1e-5,
    entropy_coef=0.05,
    per_device_train_batch_size=4,
    num_generations=4,
    report_to="wandb",
    num_train_epochs=2,
    logging_steps=1,
    log_completions=True,
    max_completion_length=2048
)

In [49]:
system_prompts = """
Your job is to design a prompt for wild, unconventional, creative, and anti-design ad promotion posters for the object. The image should be a designed poster, not regular photo. You should output both a positive prompt and negative prompt, splited by '%%'. The negative prompt should describe thing that can push the poster into the wild domain from conventional ones. The negative prompt should not use negation.
Format:
positive_prompt %% negative_prompt.
"""

question = "Write a single image generation prompt to generate wild, creative, and anti-design promotion posters for <name>"

from datasets import Dataset
names = [
    "orange juice", "coke", "bed", "laptop", "LLMs", "pizza", "robots", "orange", "coffee", "phone",
    "TV", "chair", "lamp", "car", "cup", "toilet", "bathtub", "GPU", "SD Card", "SSD",
    "earphone", "train tickets", "movie tickets", "bell", "taco", "french fries",

    # food & drink
    "sushi", "ramen", "dumplings", "burger", "hot dog", "croissant", "bagel", "donut", "ice cream", "chocolate bar",
    "bubble tea", "green tea", "beer", "wine", "whiskey", "energy drink", "sparkling water", "milk", "yogurt", "cereal",
    "instant noodles", "kimchi", "tofu", "avocado", "banana", "watermelon", "strawberries", "honey", "hot sauce", "ketchup",
    "popcorn", "potato chips", "gummy bears", "birthday cake", "frozen pizza", "canned soup", "olive oil", "salt", "eggs", "bread",

    # tech
    "smartwatch", "tablet", "keyboard", "mouse", "monitor", "webcam", "router", "power bank", "USB cable", "phone charger",
    "VR headset", "game console", "drone", "3D printer", "smart speaker", "e-reader", "hard drive", "CPU", "RAM", "motherboard",
    "cloud storage", "VPN", "password manager", "antivirus software", "operating system", "programming language", "API", "chatbot", "search engine", "social media app",
    "dating app", "food delivery app", "ride-hailing app", "streaming service", "podcast", "NFT", "cryptocurrency", "QR code", "Wi-Fi", "5G",

    # home & furniture
    "sofa", "desk", "bookshelf", "mattress", "pillow", "blanket", "curtains", "rug", "mirror", "wardrobe",
    "refrigerator", "microwave", "toaster", "blender", "kettle", "rice cooker", "air fryer", "dishwasher", "washing machine", "vacuum cleaner",
    "air conditioner", "space heater", "ceiling fan", "candle", "houseplant", "door lock", "doorbell", "alarm clock", "wall clock", "trash can",
    "toothbrush", "toothpaste", "shampoo", "soap", "toilet paper", "towel", "razor", "hair dryer", "deodorant", "sunscreen",

    # fashion & personal
    "sneakers", "high heels", "sunglasses", "hat", "umbrella", "backpack", "handbag", "wallet", "wristwatch", "jeans",
    "T-shirt", "hoodie", "socks", "underwear", "raincoat", "perfume", "lipstick", "nail polish", "glasses", "contact lenses",

    # transport & travel
    "bicycle", "electric scooter", "motorcycle", "bus pass", "airplane tickets", "subway", "taxi", "hotel", "suitcase", "passport",
    "gas station", "parking lot", "electric car", "skateboard", "cruise ship", "camping tent", "road trip", "theme park", "museum", "zoo",

    # services & abstract
    "insurance", "bank account", "credit card", "mortgage", "gym membership", "yoga class", "therapy", "dentist", "haircut", "laundromat",
    "university", "online course", "library", "newspaper", "dictionary", "calculator", "tax filing", "election", "job interview", "retirement plan",
    "wedding", "funeral service", "birthday party", "dating", "friendship", "sleep", "time", "nothing", "Monday", "weekend",

    # objects & misc
    "pencil", "eraser", "notebook", "stapler", "scissors", "glue", "paperclip", "sticky notes", "envelope", "stamp",
    "rubber duck", "teddy bear", "lego bricks", "board game", "playing cards", "puzzle", "kite", "balloon", "fireworks", "guitar",
    "piano", "drum kit", "vinyl record", "cassette tape", "film camera", "batteries", "light bulb", "extension cord", "hammer", "duct tape",
    "brick", "cement", "rock", "cloud", "rain", "sunlight", "air", "water", "fire extinguisher", "first aid kit",

    # pets & nature
    "dog food", "cat litter", "fish tank", "bird cage", "pet insurance", "garden hose", "lawn mower", "fertilizer", "seeds", "flowers",
]

def format_prompt(prompt):
  return [
      {
          "role": "system",
          "content": system_prompts
      },
      {
          "role": "user",
          "content": prompt
      }
  ]
ds = Dataset.from_list([{"prompt": format_prompt(question.replace("<name>", name)), "product": name} for name in names] * 10)

In [50]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

processor = AutoProcessor.from_pretrained("Qwen/Qwen3.5-2B")
model = AutoModelForMultimodalLM.from_pretrained("Qwen/Qwen3.5-2B", device_map="auto")

Loading weights: 100%|██████████| 617/617 [00:00<00:00, 1396.24it/s]


In [ ]:
from datasets import load_dataset
from trl import GRPOTrainer
from trl.rewards import accuracy_reward

wandb.init()
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[format_reward, reward_func_text, reward_func_image],
    train_dataset=ds,
    args=training_args,
)
trainer.train()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


profiling/Time taken: GRPOTrainer._calculate_rewards,▁
profiling/Time taken: GRPOTrainer._prepare_inputs,▁
profiling/Time taken: GRPOTrainer.format_reward,▁
profiling/Time taken: GRPOTrainer.reward_func_image,▁
profiling/Time taken: GRPOTrainer.reward_func_text,▁
profiling/Time taken: GRPOTrainer.transformers.generate,▁
train/global_step,▁▁▁▁▁▁▁
profiling/Time taken: GRPOTrainer._calculate_rewards,41.13974
profiling/Time taken: GRPOTrainer._prepare_inputs,117.44437
profiling/Time taken: GRPOTrainer.format_reward,9e-05
profiling/Time taken: GRPOTrainer.reward_func_image,40.47923


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


[0.0, 0.1587, 0.0034583333333333332, 0.23833333333333337]


100%|██████████| 50/50 [00:08<00:00,  5.84it/s]


...

╭──────────────────────────────────────────────────── Step 3 ─────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt             ┃ Completion          ┃ format_reward ┃ reward_func_text ┃ reward_func_image ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ SYSTEM             │ ASSISTANT           │          2.00 │             0.41 │              1.36 │      0.66 │ │
│ │                    │ drawn sauerkraut    │               │                  │                   │           │ │
│ │ Your job is to     │ swirling hot dog in │               │                  │                   │           │ │
│ │ design a prompt    │ standing stick      │               │                  │                   │           │ │
│ │ for wild,          │ confetti            │               │                  │                   │           │ │
│ │ unconventional,    │                     │               │                  │                   │           │ │
│ │ creative, and      │                     │               │                  │                   │           │ │
│ │ anti-design ad     │                     │               │                  │                   │           │ │
│ │ promotion posters  │                     │               │                  │                   │           │ │
│ │ for the object.    │                     │               │                  │                   │           │ │
│ │ The image should   │                     │               │                  │                   │           │ │
│ │ be a designed      │                     │               │                  │                   │           │ │
│ │ poster, not        │                     │               │                  │                   │           │ │
│ │ regular photo. You │                     │               │                  │                   │           │ │
│ │ should output both │                     │               │                  │                   │           │ │
│ │ a positive prompt  │                     │               │                  │                   │           │ │
│ │ and negative       │                     │               │                  │                   │           │ │
│ │ prompt, splited by │                     │               │                  │                   │           │ │
│ │ '%%'. The negative │                     │               │                  │                   │           │ │
│ │ prompt should      │                     │               │                  │                   │           │ │
│ │ describe thing     │                     │               │                  │                   │           │ │
│ │ that can push the  │                     │               │                  │                   │           │ │
│ │ poster into the    │                     │               │                  │                   │           │ │
│ │ wild domain from   │                     │               │                  │                   │           │ │
│ │ conventional ones. │                     │               │                  │                   │           │ │
│ │ The negative       │                     │               │                  │                   │           │ │
│ │ prompt should not  │                     │               │                  │                   │           │ │
│ │ use negation.      │                     │               │                  │                   │           │ │
│ │ Format:            │                     │               │                  │                   │           │ │
│ │ positive_prompt %% │                     │               │                  │                   │           │ │
│ │ negative_prompt.   │                     │          

[0.005725, 0.040191666666666674, 0.15773333333333334, 0.22270000000000004]


100%|██████████| 50/50 [00:08<00:00,  5.82it/s]


[np.float32(0.90799516), np.float32(0.84979737), np.float32(0.7674301), np.float32(0.8261624)]
[0.023333333333333334, 0.375, 0.7016666666666667, 0.45166666666666666]


╭──────────────────────────────────────────────────── Step 4 ─────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt             ┃ Completion          ┃ format_reward ┃ reward_func_text ┃ reward_func_image ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ SYSTEM             │ ASSISTANT           │          1.00 │             0.01 │              0.93 │     -1.28 │ │
│ │                    │ crowbar %%          │               │                  │                   │           │ │
│ │ Your job is to     │ magicalwand         │               │                  │                   │           │ │
│ │ design a prompt    │                     │               │                  │                   │           │ │
│ │ for wild,          │ This prompt meets   │               │                  │                   │           │ │
│ │ unconventional,    │ your criteria in    │               │                  │                   │           │ │
│ │ creative, and      │ three ways:         │               │                  │                   │           │ │
│ │ anti-design ad     │ **identifying the   │               │                  │                   │           │ │
│ │ promotion posters  │ specific niche      │               │                  │                   │           │ │
│ │ for the object.    │ subject**, creating │               │                  │                   │           │ │
│ │ The image should   │ a visual style      │               │                  │                   │           │ │
│ │ be a designed      │ distinct from       │               │                  │                   │           │ │
│ │ poster, not        │ generic medical     │               │                  │                   │           │ │
│ │ regular photo. You │ posters, and using  │               │                  │                   │           │ │
│ │ should output both │ the phrase **"wacky │               │                  │                   │           │ │
│ │ a positive prompt  │ banana"** to act as │               │                  │                   │           │ │
│ │ and negative       │ a literal           │               │                  │                   │           │ │
│ │ prompt, splited by │ embodiment of the   │               │                  │                   │           │ │
│ │ '%%'. The negative │ "wild,              │               │                  │                   │           │ │
│ │ prompt should      │ unconventional"     │               │                  │                   │           │ │
│ │ describe thing     │ world you want to   │               │                  │                   │           │ │
│ │ that can push the  │ enter.              │               │                  │                   │           │ │
│ │ poster into the    │                     │               │                  │                   │           │ │
│ │ wild domain from   │ Interestingly, the  │               │                  │                   │           │ │
│ │ conventional ones. │ phrase "**fish      │               │                  │                   │           │ │
│ │ The negative       │ tails**" gets       │               │                  │                   │           │ │
│ │ prompt should not  │ referenced, even    │               │                  │                   │           │ │
│ │ use negation.      │ though it doesn't   │               │                  │                   │           │ │
│ │ Format:            │ fit the cute theme; │               │                  │                   │           │ │
│ │ positive_prompt %% │ it serves as a      │               │                  │                   │           │ │
│ │ negative_prompt.   │ playful disregard   │          

In [ ]:
# plot bank vectors to see if it expands